# 3.1 Merge individual and panel datasets

This notebook does the following:
    - Calculates per equipment type and total electricity consumption for BL and EL
    - Merges the individual data with the electricity data
    - Cleans up and saves the dataset

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os

In [2]:
# Load data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_5.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_5.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Calculate per equipment type and total electricity use (per year) for each labgroupid

In [3]:
# Aggregate electricity use by labgroupid × survey × equipment type
elec_by_type = (
    equipment
    .groupby(["labgroupid", "survey", "equipment"])["electricity_use_kwh_year"]
    .sum()
    .reset_index()
)

# Pivot equipment types into columns → one row per labgroupid × survey
elec_long = elec_by_type.pivot_table(
    index=["labgroupid", "survey"],
    columns="equipment",
    values="electricity_use_kwh_year",
    aggfunc="sum",
    fill_value=0
).reset_index()
elec_long.columns.name = None
eq_types = [c for c in elec_long.columns if c not in ("labgroupid", "survey")]
elec_long.rename(columns={c: f"annual_electricity_{c}" for c in eq_types}, inplace=True)

# Add total across all equipment types
eq_cols = [f"annual_electricity_{c}" for c in eq_types]
elec_long["annual_electricity_total"] = elec_long[eq_cols].sum(axis=1)

print("Rows by survey wave:", elec_long["survey"].value_counts().to_dict())

Rows by survey wave: {'BL': 138, 'EL': 109}


In [4]:
# Merge static lab info into long-format electricity data
# Labs without EL equipment data will have no EL row (naturally dropped)
labs = elec_long.merge(labs, on="labgroupid", how="left")
print(f"Final dataset: {len(labs)} rows")
print(labs["survey"].value_counts())

Final dataset: 247 rows
survey
BL    138
EL    109
Name: count, dtype: int64


In [5]:
# Check for any missing values in the new electricity columns
new_elec_cols = [c for c in labs.columns if c.startswith("annual_electricity_")]
missing_elec_cols = labs[new_elec_cols].isna().sum()
print("Missing values in new electricity columns:")
print(missing_elec_cols[missing_elec_cols > 0])

Missing values in new electricity columns:
Series([], dtype: int64)


## Clean up and save processed data

In [6]:
# Create "treated" variable: 1 if "Treatment Status" == "treatment"
labs["treated"] = (labs["Treatment Status"] == "treatment").astype(int)

In [7]:
# Order cols
attitude_cols = [c for c in labs.columns if c.startswith("attitude_q_")]
calc_cols = [c for c in labs.columns if c.startswith("calc_")]
cert_cols = [c for c in labs.columns if c.startswith(("bronze_cert", "silver_cert", "gold_cert"))]
elec_cols = [c for c in labs.columns if c.startswith("annual_electricity_")]

cols = [
    "labgroupid",
    "survey",
    "treated",
    "enum_id",
    "faculty",
    "institute_id",
    "survey_date_bl",
    "survey_date_el",
    "no_researchers",
    *elec_cols,
    *attitude_cols,
    *calc_cols,
    *cert_cols,
]

In [8]:
# Order cols (with cols at beginning in the desired order, and the rest in original order)
labs = labs[cols + [c for c in labs.columns if c not in cols]]

# Save processed data (with cols at beginning in the desired order)
labs.to_csv(config.CLEAN_DATA / "final_dataset.csv", index=False)